In [20]:
!wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv

--2026-10-05 18:59:46--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 635180 (620K) [text/plain]
Saving to: ‘car_fuel_efficiency_2026.csv.1’

car_fuel_efficiency 100%[===================>] 620.29K  --.-KB/s    in 0.06s   

2026-10-05 18:59:46 (10.2 MB/s) - ‘car_fuel_efficiency_2026.csv.1’ saved [635180/635180]



In [21]:
import pandas as pd
df= pd.read_csv('car_fuel_efficiency_2026.csv')
df.head(5)

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [26]:
df.isnull().sum()

engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64

In [23]:
df['horsepower'].median()

np.float64(254.0)

In [24]:
import numpy as np
df=df[['engine_displacement',
'horsepower',
'vehicle_weight',
'model_year',
'fuel_efficiency_mpg']]
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

In [25]:
def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]
    
def predict(X, w0, w):
    return w0 + X.dot(w)
    
def rmse(y, y_pred):
    error = y - y_pred
    mse = (error ** 2).mean()
    return np.sqrt(mse)

In [27]:
df_train_0 = df_train.copy()
df_val_0 = df_val.copy()

df_train_0['horsepower'] = df_train_0['horsepower'].fillna(0)
df_val_0['horsepower'] = df_val_0['horsepower'].fillna(0)

X_train = df_train_0.drop('fuel_efficiency_mpg', axis=1).values
y_train = df_train_0['fuel_efficiency_mpg'].values

X_val = df_val_0.drop('fuel_efficiency_mpg', axis=1).values
y_val = df_val_0['fuel_efficiency_mpg'].values

w0, w = train_linear_regression(X_train, y_train)

y_pred = predict(X_val, w0, w)

rmse_0 = rmse(y_val, y_pred)

print(round(rmse_0, 3))

2.205


In [28]:
df_train_mean = df_train.copy()
df_val_mean = df_val.copy()

mean_hp = df_train_mean['horsepower'].mean()

df_train_mean['horsepower'] = df_train_mean['horsepower'].fillna(mean_hp)
df_val_mean['horsepower'] = df_val_mean['horsepower'].fillna(mean_hp)

X_train = df_train_mean.drop('fuel_efficiency_mpg', axis=1).values
y_train = df_train_mean['fuel_efficiency_mpg'].values

X_val = df_val_mean.drop('fuel_efficiency_mpg', axis=1).values
y_val = df_val_mean['fuel_efficiency_mpg'].values

w0, w = train_linear_regression(X_train, y_train)

y_pred = predict(X_val, w0, w)

rmse_mean = rmse(y_val, y_pred)

print(round(rmse_mean, 3))

2.202


In [31]:
def train_linear_regression_reg(X, y, r=0.001):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])

    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]
    
df_train = df_train.copy()
df_val = df_val.copy()

df_train['horsepower'] = df_train['horsepower'].fillna(0)
df_val['horsepower'] = df_val['horsepower'].fillna(0)

X_train = df_train.drop('fuel_efficiency_mpg', axis=1).values
y_train = df_train['fuel_efficiency_mpg'].values

X_val = df_val.drop('fuel_efficiency_mpg', axis=1).values
y_val = df_val['fuel_efficiency_mpg'].values

for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    w0, w = train_linear_regression_reg(X_train, y_train, r=r)
    y_pred = predict(X_val, w0, w)

    score = rmse(y_val, y_pred)

    print(r, round(score, 4))

0 2.2053
0.01 2.2058
0.1 2.2241
1 2.3492
5 2.4094
10 2.4195
100 2.4292


In [32]:
scores = []

for seed in range(10):

    n = len(df)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test

    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)

    df_train = df.iloc[idx[:n_train]]
    df_val = df.iloc[idx[n_train:n_train + n_val]]

    # Fill missing horsepower with 0
    df_train = df_train.copy()
    df_val = df_val.copy()

    df_train['horsepower'] = df_train['horsepower'].fillna(0)
    df_val['horsepower'] = df_val['horsepower'].fillna(0)

    X_train = df_train.drop('fuel_efficiency_mpg', axis=1).values
    y_train = df_train['fuel_efficiency_mpg'].values

    X_val = df_val.drop('fuel_efficiency_mpg', axis=1).values
    y_val = df_val['fuel_efficiency_mpg'].values

    w0, w = train_linear_regression(X_train, y_train)

    y_pred = predict(X_val, w0, w)

    score = rmse(y_val, y_pred)

    scores.append(score)

    print(seed, score)

std = np.std(scores)

print("std =", round(std, 3))

0 2.239204143046126
1 2.2021128210838845
2 2.1634197787530947
3 2.2043588768539224
4 2.2018800943311554
5 2.2457851509085134
6 2.2697212079524043
7 2.190794599933443
8 2.216611201686444
9 2.2070365928178957
std = 0.029


In [33]:
df=df[['engine_displacement',
'horsepower',
'vehicle_weight',
'model_year',
'fuel_efficiency_mpg']]
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(9)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

df_train_full = pd.concat([df_train, df_val])

df_train_full = df_train_full.copy()
df_test = df_test.copy()

df_train_full['horsepower'] = df_train_full['horsepower'].fillna(0)
df_test['horsepower'] = df_test['horsepower'].fillna(0)

X_train = df_train_full.drop('fuel_efficiency_mpg', axis=1).values
y_train = df_train_full['fuel_efficiency_mpg'].values

X_test = df_test.drop('fuel_efficiency_mpg', axis=1).values
y_test = df_test['fuel_efficiency_mpg'].values



In [34]:
def train_linear_regression_reg(X, y, r=0):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)

    XTX = XTX + r * np.eye(XTX.shape[0])

    XTX_inv = np.linalg.inv(XTX)

    w = XTX_inv.dot(X.T).dot(y)

    return w[0], w[1:]
w0, w = train_linear_regression_reg(
    X_train,
    y_train,
    r=0.001
)

y_pred = predict(X_test, w0, w)

score = rmse(y_test, y_pred)

print(score)

2.235827747191731
